In [ ]:
# imports and setup
from pathlib import Path
import sys
ROOT_PATH = (Path.cwd().parents[1]).as_posix()

if ROOT_PATH not in sys.path:
    sys.path.insert(0, ROOT_PATH)
import torch
import matplotlib.pyplot as plt 
from src.utils.drw import DRWManager
from src.utils.cyclic_detector import CyclicDetector
from src.utils.light_curve_sin_dataset import LightCurvesSinDataset
from src.utils.noise_models import AR1UniformParameterDistribution, AR2CorrelatedParameterDistribution, AR1NoiseModel, AR2NoiseModel, StupidAR1NoiseModel
from src.utils.rendering import plot_rocs_from_items, plot_pvalue_histograms_from_items
from tqdm import tqdm

drw_manager = DRWManager()
ar1_parameter_distribution = AR1UniformParameterDistribution()
ar2_parameter_distribution = AR2CorrelatedParameterDistribution()
coeffs = ar2_parameter_distribution.sample(10)
curves = drw_manager.simulate_from_params(coeffs)
estimated_coeffs = drw_manager.estimate_parameters(curves, 2)

In [ ]:
# experiment

drw_manager = DRWManager(
    sigma=0.2,
    observation_noise = 0.0
)

ar1_noise_model = AR1NoiseModel(drw_manager)
ar2_noise_model = AR2NoiseModel(drw_manager)
ar2_noise_model_oracle = AR2NoiseModel(drw_manager, oracle_mode=True)
dataset = LightCurvesSinDataset(
    ar2_noise_model,
    snr_rms_range=(0.6, 1.0),
    cycle_length_range=(30, 100),
    dataset_size=2000
)
num_simulated_light_curves = 300
batch_size = 500

detector_ar1 = CyclicDetector(dataset, ar1_noise_model, num_simulated_light_curves=num_simulated_light_curves)
detector_ar2 = CyclicDetector(dataset, ar2_noise_model, num_simulated_light_curves=num_simulated_light_curves)
detector_ar2_oracle = CyclicDetector(dataset, ar2_noise_model_oracle, num_simulated_light_curves=num_simulated_light_curves)
ar2_oracle_p_values, ar2_oracle_labels = detector_ar2_oracle.get_p_value_for_dataset(batch_size=batch_size)
ar2_p_values, ar2_labels = detector_ar2.get_p_value_for_dataset(batch_size=batch_size)
ar1_p_values, ar1_labels = detector_ar1.get_p_value_for_dataset(batch_size=batch_size)

plot_rocs_from_items([("ar1", ar1_p_values, ar1_labels), ("ar2", ar2_p_values, ar2_labels), ("ar2 oracle", ar2_oracle_p_values, ar2_oracle_labels)])
plt.show()
plot_pvalue_histograms_from_items([("ar1", ar1_p_values, ar1_labels), ("ar2", ar2_p_values, ar2_labels), ("ar2 oracle", ar2_oracle_p_values, ar2_oracle_labels)])
plt.show()

In [ ]:
### AR1 vs stupid AR1
drw_manager = DRWManager(
    sigma=0.2,
    observation_noise = 0.0
)

ar1_noise_model = AR1NoiseModel(drw_manager)
ar1_oracle_noise_model = AR1NoiseModel(drw_manager, oracle_mode=True)
stupid_ar1_noise_model = StupidAR1NoiseModel(drw_manager)
dataset = LightCurvesSinDataset(
    ar1_noise_model,
    snr_rms_range=(0.5, 0.8),
    cycle_length_range=(30, 100),
    dataset_size=2000
)
batch_size = 500
num_simulated_light_curves = 300
detector_ar1 = CyclicDetector(dataset, ar1_noise_model, num_simulated_light_curves=num_simulated_light_curves)
detector_ar1_stupid = CyclicDetector(dataset, stupid_ar1_noise_model, num_simulated_light_curves=num_simulated_light_curves)
detector_ar1_oracle = CyclicDetector(dataset, ar1_oracle_noise_model, num_simulated_light_curves=num_simulated_light_curves)
ar1_stupid_p_values, ar1_stupid_labels = detector_ar1_stupid.get_p_value_for_dataset(batch_size=batch_size)
ar1_p_values, ar1_labels = detector_ar1.get_p_value_for_dataset(batch_size=batch_size)
ar1_oracle_p_values, ar1_oracle_labels = detector_ar1_oracle.get_p_value_for_dataset(batch_size=batch_size)

plot_rocs_from_items([("ar1", ar1_p_values, ar1_labels), ("ar1 stupid", ar1_stupid_p_values, ar1_stupid_labels), ("ar1 oracle", ar1_oracle_p_values, ar1_oracle_labels)])
plt.show()
plot_pvalue_histograms_from_items([("ar1", ar1_p_values, ar1_labels), ("ar1 stupid", ar1_stupid_p_values, ar1_stupid_labels), ("ar1 oracle", ar1_oracle_p_values, ar1_oracle_labels)])
plt.show()

In [ ]:
## datasets
import os 
import numpy as np
drw_manager = DRWManager(
    sigma=0.2,
    observation_noise = 0.0
)
dataset_size = 100_000
num_samples = 2048
ar1_noise_model = AR1NoiseModel(drw_manager, num_samples=num_samples)
ar2_noise_model = AR2NoiseModel(drw_manager, num_samples=num_samples)

ar1_curves, ar1_coeffs = ar1_noise_model.generate_curves(dataset_size)
ar1_dataset = LightCurvesSinDataset(
    ar1_noise_model,
    snr_rms_range=(0.6, 1.0),
    cycle_length_range=(30, 100),
    dataset_size=dataset_size
)
ar1_outlier_curves = ar1_dataset.curves[ar1_dataset.sin_wave_mask]

ar1_dataset_folder = "../../datasets/ar1"
os.makedirs(ar1_dataset_folder, exist_ok=True)
np.save(os.path.join(ar1_dataset_folder, "curves.npy"), ar1_curves.cpu().numpy())
np.save(os.path.join(ar1_dataset_folder, "outliers.npy"), ar1_outlier_curves.cpu().numpy())
np.save(os.path.join(ar1_dataset_folder, "coeffs.npy"), ar1_coeffs.cpu().numpy())

ar2_curves, ar2_coeffs = ar2_noise_model.generate_curves(dataset_size)
ar2_dataset = LightCurvesSinDataset(
    ar2_noise_model,
    snr_rms_range=(0.6, 1.0),
    cycle_length_range=(30, 100),
    dataset_size=dataset_size
)
ar2_outlier_curves = ar2_dataset.curves[ar2_dataset.sin_wave_mask]
ar2_dataset_folder = "../../datasets/ar2"
os.makedirs(ar2_dataset_folder, exist_ok=True)
np.save(os.path.join(ar2_dataset_folder, "curves.npy"), ar2_curves.cpu().numpy())
np.save(os.path.join(ar2_dataset_folder, "outliers.npy"), ar2_outlier_curves.cpu().numpy())
np.save(os.path.join(ar2_dataset_folder, "coeffs.npy"), ar2_coeffs.cpu().numpy())
